# MongoDB Query Optimization 

**Datasets:** sample_mflix, sample_airbnb, sample_supplies

## Objectives
- Identify slow queries and bottlenecks
- Design and implement effective indexes
- Optimize aggregation pipelines
- Use MongoDB profiler and explain plans
- Apply query optimization best practices
- Understand query execution internals

---

## Setup & Connection

In [ ]:
# Install required packages
!pip install pymongo dnspython matplotlib pandas

In [ ]:
!pip install pymongo dotenv

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

if not os.environ.get("MONGO_CONNECTION_STRING"):
    print("Connection string for MONGO is not set. Please check your .env file.")
else:
    print("MONGO_CONNECTION_STRING loaded successfully.")

In [ ]:
import pymongo

MONGO_CONNECTION_STRING = os.environ.get("MONGO_CONNECTION_STRING")
mongo_client = pymongo.MongoClient(MONGO_CONNECTION_STRING)

mflix_db = mongo_client.sample_mflix
airbnb_db = mongo_client.sample_airbnb

movies = mflix_db.movies
comments = mflix_db.comments
users = mflix_db.users
listings = airbnb_db.listingsAndReviews

print("✅ Connected to MongoDB Atlas!")
print(f"Movies: {movies.count_documents({}):,}")
print(f"Comments: {comments.count_documents({}):,}")
print(f"Listings: {listings.count_documents({}):,}")

### Helper Functions for Performance Testing



In [ ]:
import time

def time_query(collection, query, name="Query", limit=None):
    """Time a query execution and return results"""
    start = time.time()
    cursor = collection.find(query)
    if limit:
        cursor = cursor.limit(limit)
    results = list(cursor)  # Force execution
    elapsed = (time.time() - start) * 1000  # Convert to ms
    print(f"{name}: {elapsed:.2f}ms ({len(results)} documents)")
    return results, elapsed

def time_aggregation(collection, pipeline, name="Aggregation"):
    """Time an aggregation pipeline"""
    start = time.time()
    results = list(collection.aggregate(pipeline))
    elapsed = (time.time() - start) * 1000
    print(f"{name}: {elapsed:.2f}ms ({len(results)} documents)")
    return results, elapsed

def explain_query(collection, query, projection=None):
    """Get detailed explain plan for a query"""
    cursor = collection.find(query, projection)
    explain = cursor.explain()
    return explain

def print_explain_summary(explain):
    """Print a human-readable summary of explain output"""
    if 'executionStats' in explain:
        stats = explain['executionStats']
        print(f"\nExecution Stats:")
        print(f"Execution time: {stats.get('executionTimeMillis', 0)}ms")
        print(f"Documents examined: {stats.get('totalDocsExamined', 0):,}")
        print(f"Documents returned: {stats.get('nReturned', 0):,}")
        
        examined = stats.get('totalDocsExamined', 0)
        returned = stats.get('nReturned', 0)
        if examined > 0:
            ratio = returned / examined
            print(f"Efficiency ratio: {ratio:.2%}")
            if ratio < 0.1:
                print("  ⚠️  LOW EFFICIENCY - Consider adding an index!")
    
    if 'queryPlanner' in explain:
        planner = explain['queryPlanner']
        winning_plan = planner.get('winningPlan', {})
        stage = winning_plan.get('stage', 'UNKNOWN')
        
        print(f"\nQuery Plan:")
        print(f"  Stage: {stage}")
        
        if stage == 'COLLSCAN':
            print("  ⚠️  COLLECTION SCAN - No index used!")
        elif 'inputStage' in winning_plan:
            input_stage = winning_plan['inputStage']
            if 'indexName' in input_stage:
                print(f"  ✅ Using index: {input_stage['indexName']}")

print("Helper functions loaded!")

---
## 1. Analyze Query Performance

The **COLLSCAN** stage indicates a collection scan is perform, not using any indexes.

The **IXSCAN** stage indicates the query is using an index and what index is being selected.

In [ ]:
# Find movies without using indexes
print("-"*50)
query = {"imdb.rating": {"$gt": 7.5}}

results, elapsed = time_query(movies, query, "Query without index")

explain = explain_query(movies, query)

print_explain_summary(explain)

In [ ]:
# Find movies without using indexes
print("-"*50)
query = {"year": {"$gt": 2015}}

results, elapsed = time_query(movies, query, "Query with index")

explain = explain_query(movies, query)

print_explain_summary(explain)

### Exercise 1: Identify Performance Issues

Analyze these queries and identify which ones would be slow and why.

In [ ]:
# Query A: Find by exact title
query_a = {"title": "The Matrix"}
explain = explain_query(movies, query_a, "query a")
print_explain_summary(explain)

# Query B: Find by year range and genre
query_b = {"price": {"$gte": 250, "$lte": 255}}
explain = explain_query(listings, query_b, "query b")
print_explain_summary(explain)

# Query C: Find by rating and sort by year
query_c = {"genres": "Action"}
explain = explain_query(movies, query_c, "query c")
print_explain_summary(explain)

---
## 2. Index Design and Implementation

### 2.1 Single Field Indexes

In [ ]:
from pymongo import ASCENDING, DESCENDING

# Create a single field index on price
print("Creating index on 'price' field...")
try:
    listings.create_index([("price", ASCENDING)], name="price_1")
    print("✅ Index created successfully!")
except Exception as e:
    print(f"Index may already exist: {e}")

# Test the same query with index
query = {"price": {"$gte": 250}}

print("\nQuery WITH index:")
results, elapsed = time_query(listings, query, "With Index")

explain = explain_query(listings, query)
print_explain_summary(explain)

### 2.2 Compound Indexes

Compound indexes can optimize queries with multiple fields.

In [ ]:
from datetime import datetime
utc_datetime = datetime(2015, 7, 27, 0, 0, 0)

# Query with multiple fields
query = {"released": {"$gt": utc_datetime}, "imdb.rating": {"$gt": 7.5}}


print("Before compound index:")
explain = explain_query(movies, query)
print_explain_summary(explain)

# Create compound index
print("\nCreating compound index on (year, imdb.rating)...")
try:
    movies.create_index(
        [("released", ASCENDING), ("imdb.rating", DESCENDING)],
        name="released_1_rating_-1"
    )
    print("✅ Compound index created!")
except Exception as e:
    print(f"Index may already exist: {e}")

print("\nAfter compound index:")
explain = explain_query(movies, query)
print_explain_summary(explain)

### 2.3 Index Field Order

The **ESR** rule is MongoDB’s guideline for how to order fields in a compound index to get the most efficient query performance. When designing an index:
- **E** Put equality fields first (fields queried with = or \\$in)
- **S** Then put sorting fields (fields used in sort())
- **R** Then put range fields (fields queried with \\$gt, \\$lt, \\$gte, \\$lte, etc.).

In [ ]:
# ESR Rule: Equality, Sort, Range
# Example: Find movies by genre, sorted by year, with rating > 7
query = {
    "genres": "Action",           # E - Equality
    "imdb.rating": {"$gt": 7}    # R - Range
}
sort = [("year", DESCENDING)]     # S - Sort

print("\nGood index order: (genres, year, imdb.rating)")
print("   Follows ESR rule")

# Create the optimized index
try:
    movies.create_index(
        [("genres", ASCENDING), ("year", DESCENDING), ("imdb.rating", DESCENDING)],
        name="genres_1_year_-1_rating_-1"
    )
    print("\n✅ Optimized compound index created!")
except Exception as e:
    print(f"\nIndex may already exist: {e}")

cursor = movies.find(query).sort(sort).limit(10)
explain = cursor.explain()
print_explain_summary(explain)

### 2.4 Covered Queries

All fields in projection are in the index

In [ ]:
# Covered query
query = {"year": 2015}
projection = {"_id": 0, "year": 1}  # Only request indexed fields

print("Testing Covered Query:")
print("   Query only uses fields in the index")
print("   MongoDB can answer WITHOUT reading documents!")

cursor = movies.find(query, projection)
explain = cursor.explain()

winning_plan = explain.get('queryPlanner', {}).get('winningPlan', {})
if winning_plan.get('stage') == 'PROJECTION_COVERED':
    print("\n✅ COVERED QUERY! Ultra-fast execution.")
else:
    print(f"\nQuery plan stage: {winning_plan.get('stage')}")
    
print_explain_summary(explain)

### 2.5 Multikey Indexes (Arrays)

Multikey indexes collect and sort data from fields containing array values. Multikey indexes improve performance for queries on array fields. Each element in the array becomes a separate index key.

In [ ]:
# Indexes on array fields
print("Creating multikey index on 'genres'...")
try:
    movies.create_index([("genres", ASCENDING)], name="genres_1")
    print("✅ Multikey index created!")
except Exception as e:
    print(f"Index may already exist: {e}")

# Query array fields
query = {"genres": "Comedy"}

print("\nQuerying array field with index:")
explain = explain_query(movies, query)
print_explain_summary(explain)

print("\n⚠️ Important: Compound indexes can have at most ONE array field")

### 2.6 Text Indexes for Search

In [ ]:
# Create text index for full-text search
print("Creating text index on title and plot...")
try:
    movies.create_index(
        [("title", "text"), ("plot", "text")],
        name="text_search_idx",
        default_language="english"
    )
    print("✅ Text index created!")
except Exception as e:
    print(f"Index may already exist: {e}")

# Use text search
results = movies.find(
    {"$text": {"$search": "space adventure alien"}},
    {"title": 1, "plot": 1, "score": {"$meta": "textScore"}}
).sort([("score", {"$meta": "textScore"})]).limit(5)

print("\n🔍 Text search results:")
for movie in results:
    print(f"  {movie['title']} (score: {movie.get('score', 0):.2f})")

### Exercise 2: Design Optimal Indexes

For each query pattern, design the best index. Use ESR rule

In [ ]:
# Scenario 1: Find listings by country, sorted by price
query_1 = {"address.country": "Spain"}
sort_1 = [("price", ASCENDING)]

# Scenario 2: Find listings with 2+ bedrooms, 2+ beds, sorted by reviews
query_2 = {"bedrooms": {"$gte": 2}, "beds": {"$gte": 2}}
sort_2 = [("number_of_reviews", DESCENDING)]

# Scenario 3: Find by property type, price range, high rating
query_3 = {
    "property_type": "Apartment",
    "price": {"$lte": 100},
    "review_scores.review_scores_rating": {"$gte": 90}
}

# Test your indexes
# listings.create_index([...])

---
## 3. Aggregation Pipeline Optimization

### 3.1 Pipeline Stage Order

In [ ]:
# Bad pipeline: Filter late
bad_pipeline = [
    {"$project": {"title": 1, "year": 1, "genres": 1, "imdb": 1}},
    {"$unwind": "$genres"},
    {"$match": {"year": {"$gte": 2010}}},  # Too late!
    {"$group": {
        "_id": "$genres",
        "count": {"$sum": 1}
    }}
]

print("❌ Bad pipeline (filter late):")
results, bad_time = time_aggregation(movies, bad_pipeline, "Bad")

# Good pipeline: Filter early
good_pipeline = [
    {"$match": {"year": {"$gte": 2010}}},  # Filter FIRST!
    {"$project": {"title": 1, "year": 1, "genres": 1, "imdb": 1}},
    {"$unwind": "$genres"},
    {"$group": {
        "_id": "$genres",
        "count": {"$sum": 1}
    }}
]

print("\n✅ Good pipeline (filter early):")
results, good_time = time_aggregation(movies, good_pipeline, "Good")

print(f"\n⚡ Performance improvement: {(bad_time/good_time):.2f}x faster")

### 3.2 Using Indexes in Aggregation

In [ ]:
# Pipeline that can use indexes
pipeline = [
    {"$match": {"year": {"$gte": 2010}}},  # Can use year index
    {"$sort": {"imdb.rating": -1}},        # Can use compound index
    {"$limit": 10},
    {"$project": {"title": 1, "year": 1, "imdb.rating": 1}}
]

print("Checking if aggregation uses indexes:")

# Get explain for aggregation
explain = mflix_db.command(
    'explain',
    {'aggregate': 'movies', 'pipeline': pipeline, 'cursor': {}},
    verbosity='executionStats'
)

if 'stages' in explain:
    print("\nPipeline stages:")
    for stage in explain['stages']:
        if '$cursor' in stage:
            query_planner = stage['$cursor'].get('queryPlanner', {})
            winning_plan = query_planner.get('winningPlan', {})
            print(f"  Stage: {winning_plan.get('stage', 'N/A')}")
            if 'inputStage' in winning_plan:
                input_stage = winning_plan['inputStage']
                if 'indexName' in input_stage:
                    print(f"  ✅ Using index: {input_stage['indexName']}")

### 3.3 $lookup Optimization

In [ ]:
# $lookup can be expensive - optimize with indexes
print("Creating index on comments.movie_id for $lookup...")
try:
    comments.create_index([("movie_id", ASCENDING)], name="movie_id_1")
    print("✅ Index created!")
except Exception as e:
    print(f"Index may already exist: {e}")

# Unoptimized: lookup on all movies
bad_lookup_pipeline = [
    {"$lookup": {
        "from": "comments",
        "localField": "_id",
        "foreignField": "movie_id",
        "as": "comments"
    }},
    {"$match": {"year": 2015}},  # Filter after expensive lookup!
    {"$limit": 10}
]

print("\n❌ Bad: $lookup before $match")
results, bad_lookup_time = time_aggregation(movies, bad_lookup_pipeline, "Bad Lookup")

# Optimized: filter first, then lookup
good_lookup_pipeline = [
    {"$match": {"year": 2015}},  # Filter FIRST!
    {"$limit": 10},
    {"$lookup": {
        "from": "comments",
        "localField": "_id",
        "foreignField": "movie_id",
        "as": "comments"
    }}
]

print("\n✅ Good: $match before $lookup")
results, good_lookup_time = time_aggregation(movies, good_lookup_pipeline, "Good Lookup")

print(f"\n⚡ Performance improvement: {(bad_lookup_time/good_lookup_time):.2f}x faster")

### 3.4 Avoiding Large $unwind Operations

In [ ]:
# Problem: $unwind explodes document count
print("Document explosion with $unwind:")

# Check average array size
sample = movies.find_one({"genres": {"$exists": True}})
print(f"Average genres per movie: ~{len(sample.get('genres', []))} elements")

# Bad: Unwind early
bad_unwind = [
    {"$unwind": "$genres"},  # Explodes documents 3-5x
    {"$match": {"year": {"$gte": 2010}, "genres": "Action"}},
    {"$limit": 10}
]

print("\n❌ Bad: $unwind before filtering")
results, bad_unwind_time = time_aggregation(movies, bad_unwind, "Bad Unwind")

# Good: Filter first
good_unwind = [
    {"$match": {"year": {"$gte": 2010}, "genres": "Action"}},  # Filter first
    {"$limit": 10},
    {"$unwind": "$genres"}
]

print("\n✅ Good: Filter before $unwind")
results, good_unwind_time = time_aggregation(movies, good_unwind, "Good Unwind")

print(f"\n⚡ Performance improvement: {(bad_unwind_time/good_unwind_time):.2f}x faster")

### Exercise 3: Optimize This Pipeline

In [ ]:
# Unoptimized pipeline - find and fix the issues!
unoptimized = [
    {"$project": {
        "name": 1,
        "property_type": 1,
        "price": 1,
        "bedrooms": 1,
        "amenities": 1,
        "review_scores": 1
    }},
    {"$unwind": "$amenities"},
    {"$match": {
        "bedrooms": {"$gte": 2},
        "price": {"$lte": 150}
    }},
    {"$group": {
        "_id": "$property_type",
        "avg_price": {"$avg": "$price"},
        "count": {"$sum": 1}
    }},
    {"$sort": {"avg_price": -1}}
]

print("Unoptimized pipeline:")
results, unopt_time = time_aggregation(listings, unoptimized, "Unoptimized")

# YOUR TURN: Optimize this pipeline
optimized = [
    # TODO: Add your optimized stages here
    # Hints:
    # 1. Move $match to the beginning
    # 2. $unwind?
    # 3. Project only needed fields
    # 4. Consider index usage
]

# print("\n✅ Optimized pipeline:")
# results, opt_time = time_aggregation(listings, optimized, "Optimized")
# print(f"Improvement: {(unopt_time/opt_time):.2f}x faster")